# 08. Measuring forecast errors (QLIKE and MSE)
**Part 2 — The benchmark models** starts with the scoring rules. In class, a regressor was scored with `r2_score`. For volatility forecasts the study uses **QLIKE** (main score) and **MSE**. This notebook explains them with small numbers first.

* **Needs:** `work/dataset.csv`. **Makes:** nothing new.

*Simple notebook series of the project 'This Time Is Different?'. Prepared with AI assistance (declared in `notes/ai_use.md`); a study notebook, not dissertation text.*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
folder = '/content/drive/MyDrive/DataSets/ThisTimeIsDifferent/'

import warnings
warnings.filterwarnings('ignore')     # hide warning messages

## MSE — mean squared error
The average of (realised − forecast)². You know it from class.

## QLIKE
$\text{QLIKE} = \dfrac{RV}{F} - \ln\dfrac{RV}{F} - 1$, where RV = realised variance, F = forecast. It is **0** for a perfect forecast and bigger for worse ones.

### A small example
Three weeks; the third week is a storm (realised 4). Forecaster A says 2 (too low), forecaster B says 8 (too high).

In [2]:
from pandas import Series
from numpy import log
realised = Series([1, 1, 4])
forecast_A = Series([1, 1, 2])
forecast_B = Series([1, 1, 8])

MSE_A = ((realised - forecast_A) ** 2).mean()
MSE_B = ((realised - forecast_B) ** 2).mean()
print('MSE   A =', round(MSE_A, 3), '| MSE   B =', round(MSE_B, 3))

ratio_A = realised / forecast_A
ratio_B = realised / forecast_B
QLIKE_A = (ratio_A - log(ratio_A) - 1).mean()
QLIKE_B = (ratio_B - log(ratio_B) - 1).mean()
print('QLIKE A =', round(QLIKE_A, 3), '| QLIKE B =', round(QLIKE_B, 3))

MSE   A = 1.333 | MSE   B = 5.333
QLIKE A = 0.102 | QLIKE B = 0.064


MSE says A is better; QLIKE says **B** is better. QLIKE punishes **under-prediction** — saying 'calm' before a storm — more, which is what a risk manager fears most. QLIKE also ranks forecasts correctly even when the 'true' volatility is only measured with noise (Patton, 2011). That is why it is the **main score** of the study.

## A real example: the naive forecast
The simplest possible forecast: **next week will be like last week** (the last 5-day average × 5). It is not a real model, only a reference.

In [3]:
from pandas import read_csv
dataset = read_csv(folder + 'work/dataset.csv', index_col=0, parse_dates=True)
test = dataset.loc['2007-01-01':'2023-08-31'].dropna(subset=['rv5_fwd']).copy()
y_test = test['rv5_fwd']
y_pred_naive = test['rv_w'] * 5
print(len(y_test), 'test days')

4230 test days


In [4]:
ratio = y_test / y_pred_naive
QLIKE = (ratio - log(ratio) - 1).mean()
MSE = ((y_test - y_pred_naive) ** 2).mean()
print('QLIKE (naive) =', round(QLIKE, 3))
print('MSE (naive)   =', MSE)

QLIKE (naive) = 0.802
MSE (naive)   = 3.035851842024409e-06


The MSE is a tiny number because variances are tiny (0.0004 = a normal week). That is why results tables show MSE × 1,000,000.

## The score in each window
We keep the daily QLIKE values in a column and average them per window with `groupby`:

In [5]:
test['QLIKE_naive'] = ratio - log(ratio) - 1
test.groupby('window')['QLIKE_naive'].mean().round(3)

window
Brexit referendum              1.264
COVID-19                       1.053
Global Financial Crisis        0.834
Irish sovereign debt crisis    0.802
War / energy shock 2022        0.506
calm                           0.802
Name: QLIKE_naive, dtype: float64

The naive forecast is poor in every window. (In notebook 13 you will see that in the short Brexit window it still beats the real models: it simply copied the spike of the referendum day afterwards.) The statistical test that compares two models (Diebold–Mariano) comes in notebook 13.